<a href="https://colab.research.google.com/github/Born3Life/234-tkst-bots/blob/main/ollama_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ollama + Llava на Google Colab
Запускаем Ollama с моделью llava и открываем доступ через туннель Cloudflare.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq wget curl zstd

!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'

get_ipython().system_raw('ollama serve &')
import time
time.sleep(5)
print('✅ Ollama запущен')

In [ ]:
!ollama pull llava
!ollama pull gemma3:12b
print('✅ Модели llava (зрение) и gemma3:12b (текст) готовы')

In [ ]:
import subprocess
import threading
import re
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

found_url = []

def run_tunnel():
    proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = f"{match.group()}/api/chat"
            found_url.append(url)
            print(f'\n\n*** МОЙ URL ***')
            print(f'{url}')
            print(f'***************\n')

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()

print('⏳ Ожидаю туннель... (до 30 сек)')
for _ in range(30):
    if found_url:
        break
    time.sleep(1)

if found_url:
    print(f'\n✅ Туннель готов: {found_url[0]}')
else:
    print('\n⚠️ Туннель не появился за 30 сек. Проверь вывод выше.')

In [ ]:
import threading
import requests
import json
import time

tunnel_url_local = found_url[0] if found_url else None
inference_count = 0

def keep_alive():
    global inference_count
    while True:
        time.sleep(60)
        try:
            requests.get('http://localhost:11434', timeout=5)
        except:
            pass

        inference_count += 1
        if inference_count % 5 == 0:
            # Каждые 5 мин дёргаем модель (GPU не уснёт)
            try:
                requests.post('http://localhost:11434/api/chat',
                    json={"model": "gemma3:12b", "messages": [{"role": "user", "content": "."}], "stream": False, "options": {"num_predict": 1}},
                    timeout=30)
                print('💪 GPU keep-alive')
            except:
                pass

            # Пинаем туннель изнутри (помогает cloudflared)
            if tunnel_url_local:
                try:
                    requests.get(tunnel_url_local, timeout=10)
                except:
                    pass

ka = threading.Thread(target=keep_alive, daemon=True)
ka.start()
print('✅ Keep-alive: пинг 60с + GPU каждые 5 мин')

## Не даём Colab уснуть

**1. Внешний пинг (обязательно!)**

Этот ноутбук стучит по модели каждые 5 мин, но Google может вырубить Colab при простое >90 мин.
Поставь https://cron-job.org — он будет пинать туннель из интернета каждые 5 мин:
- Register → Create Cron Job
- URL: `https://твой-туннель.trycloudflare.com`
- Every 5 minutes → Save

**2. Когда Colab отвалится (через ~12ч):**
- Runtime → Restart and run all
- Скопировать новый URL из `*** МОЙ URL ***`
- Скинуть его мне — я обновлю ботов

---
**Важно:** Если Colab отключился → боты не работают. Только перезапуск.